# 🚀 Smart Research Assistant Agent
### Powered by Amazon Nova 2 Lite on AWS Bedrock | #AmazonNova

**Amazon Nova AI Hackathon 2026** | Category: Agentic AI | Built by Devika

---

This notebook demonstrates an **autonomous AI agent** that can:
- 🔍 Research any topic using web search
- 📄 Read and analyze real PDF documents
- 📊 Perform data analysis
- 🧠 Reason about problems using Amazon Nova 2 Lite
- ✨ Synthesize intelligent answers

**Architecture:**
```
User Query → Nova Reasoning → Tool Selection → Execution → Synthesis → Answer
```

## Step 1: Install Dependencies

In [ ]:
!pip install boto3 botocore PyPDF2 reportlab -q
print("✅ All dependencies installed!")

## Step 2: Configure AWS Credentials

Enter your AWS credentials below to connect to Amazon Bedrock.

**Important:** If you hit a throttling/quota limit, set `USE_DEMO_MODE = True` below to run with simulated Nova responses (still shows the full agent architecture).

In [ ]:
import os

# ⚠️ ENTER YOUR AWS CREDENTIALS HERE
os.environ['AWS_ACCESS_KEY_ID'] = ''          # <-- Your Access Key
os.environ['AWS_SECRET_ACCESS_KEY'] = ''  # <-- Your Secret Key
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

# ============================================================
# 🔄 DEMO MODE: Set to True if you hit throttling/quota limits
# This uses simulated Nova responses to demonstrate the agent
# architecture without making API calls.
# Set to False to use REAL Amazon Nova API calls.
# ============================================================
USE_DEMO_MODE = False   # Change to True if throttled

print("✅ AWS credentials configured!")
print(f"   Region: {os.environ['AWS_DEFAULT_REGION']}")
if USE_DEMO_MODE:
    print("   ⚡ Demo mode: ON (using simulated Nova responses)")
else:
    print("   🔗 Live mode: ON (using real Amazon Nova API calls)")

## Step 3: Define the Agent Code

This cell contains the complete agent:
- **ResearchTools** — 4 tools the agent can use (web search, PDF analysis, data analysis, document fetching)
- **ResearchAgent** — The brain that reasons, decides, executes, and synthesizes
- **Throttle handling** — Automatic retry with wait if API quota is temporarily exceeded
- **Demo mode** — Simulated responses for demonstrations when API is unavailable

In [ ]:
"""
Amazon Nova AI Hackathon 2026
Project: Smart Research Assistant Agent
Category: Agentic AI
Author: Devika

This agent helps users research topics by breaking down questions,
gathering information, and providing intelligent summaries using
Amazon Nova 2 Lite for advanced reasoning.
"""

import boto3
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path
import PyPDF2

# ============================================================================
# CONFIGURATION
# ============================================================================

MODEL_ID = "amazon.nova-lite-v1:0"   # Amazon Nova 2 Lite model
AWS_REGION = "us-east-1"
MAX_RETRIES = 3                       # Retries for throttling
RETRY_WAIT = 5                        # Seconds to wait between retries

# Lazy client initialization
_bedrock_client = None

def get_bedrock_client():
    """Get or create the AWS Bedrock client (lazy initialization)."""
    global _bedrock_client
    if _bedrock_client is None:
        try:
            _bedrock_client = boto3.client(
                'bedrock-runtime',
                region_name=AWS_REGION
            )
        except Exception as e:
            print(f"\n❌ AWS Configuration Error: {e}")
            print("\nPlease check your AWS credentials in Step 2 above.")
            raise
    return _bedrock_client


# ============================================================================
# DEMO MODE — Simulated Nova responses for when API is unavailable
# ============================================================================

DEMO_RESPONSES = {
    "reasoning": {
        "agentic": "THOUGHT: The user wants to know about the current landscape of agentic AI systems. This is a broad research question that requires up-to-date information about AI trends and developments. I should search for the latest information on this topic.\nACTION: search_web\nINPUT: AI trends\nFINAL_ANSWER: I will search for the latest AI trends to provide a comprehensive overview.",
        "pdf": "THOUGHT: The user wants me to analyze a PDF document. I need to first read the PDF file to extract its content, then identify the key findings from the text. I'll use the PDF analysis tool to read the document.\nACTION: analyze_pdf\nINPUT: sample.pdf\nFINAL_ANSWER: I will read the PDF and analyze its key findings.",
        "market": "THOUGHT: The user is asking about market analysis related to AI growth. This requires data analysis rather than a simple web search. I should use the analyze_data tool with a market focus to get relevant statistics and projections.\nACTION: analyze_data\nINPUT: market\nFINAL_ANSWER: I will analyze market data to provide AI growth insights.",
        "machine learning": "THOUGHT: The user wants to learn about machine learning trends. This is a research question that I can answer by searching for the latest developments. I'll use web search to find current ML trends.\nACTION: search_web\nINPUT: machine learning\nFINAL_ANSWER: I will search for current machine learning trends.",
        "default": "THOUGHT: The user has asked a research question. I need to search for relevant information to provide an accurate and helpful answer.\nACTION: search_web\nINPUT: AI trends\nFINAL_ANSWER: I will research this topic and provide a comprehensive response."
    },
    "synthesis": {
        "agentic": "Agentic AI is one of the most rapidly evolving areas in artificial intelligence today. These systems go beyond simple prompt-response patterns — they can autonomously plan multi-step tasks, use external tools, and adapt their approach based on intermediate results.\n\nKey developments include:\n\n1. Foundation models like Amazon Nova that provide sophisticated reasoning capabilities, enabling agents to tackle complex research, analysis, and automation tasks.\n\n2. Multimodal AI is advancing vision-language tasks, allowing agents to understand and process different types of data including text, images, and documents.\n\n3. Agentic AI systems are increasingly automating complex workflows that previously required human oversight, from software development to scientific research.\n\nThe combination of powerful reasoning (like Amazon Nova 2 Lite), tool use capabilities, and multi-step planning is making these agents increasingly capable and useful across industries.",
        "pdf": "Based on the analysis of the PDF document 'The Future of Agentic AI Systems', here are the key findings:\n\n1. **Foundation Models as the Core**: Foundation models like Amazon Nova are identified as essential enablers of sophisticated reasoning in AI agents. They provide the cognitive backbone that allows agents to understand complex problems.\n\n2. **Tool Use is Critical**: The research emphasizes that for AI agents to be truly useful in the real world, they must be able to interact with external systems — APIs, files, databases, and other tools. This is what separates a simple chatbot from a real agent.\n\n3. **Multi-step Planning Boosts Performance**: The paper reports a significant 45% improvement in task completion rates when agents use multi-step planning compared to single-step approaches. This validates the importance of the reasoning-planning-execution loop.\n\nThe paper concludes that agentic AI powered by foundation models will transform workflows across industries, from research to software development — which is exactly what this agent demonstrates!",
        "market": "Based on the market analysis data, the AI industry is experiencing unprecedented growth:\n\n• The global AI market is projected to reach **$1.8 trillion by 2030**, driven by enterprise adoption of foundation models and agentic AI systems.\n\n• Key growth drivers include cloud AI services (like AWS Bedrock), automated workflow solutions, and multimodal understanding capabilities.\n\n• Companies like Amazon with their Nova foundation models are at the forefront, making advanced AI accessible through managed services.\n\n• The agentic AI segment specifically is seeing the fastest growth, as businesses recognize the value of autonomous AI systems that can reason, plan, and execute complex tasks.\n\nThis trend validates the approach of building intelligent agents that leverage powerful foundation models like Amazon Nova 2 Lite for reasoning and decision-making.",
        "machine learning": "Machine learning continues to evolve rapidly with several notable trends:\n\n1. **Foundation Models**: Large pre-trained models like Amazon Nova are becoming the standard starting point, allowing developers to build sophisticated applications without training from scratch.\n\n2. **Learning from Data**: ML algorithms that learn patterns from data without explicit programming remain at the core, but the scale and sophistication have increased dramatically.\n\n3. **Practical Applications**: ML is being applied to everything from document analysis to market prediction, making it more accessible and useful than ever.\n\nThe key takeaway is that ML is moving from a specialized tool to a foundational technology that powers intelligent systems like our research assistant agent.",
        "default": "Based on my research, here is a comprehensive analysis of the topic. The field of AI is advancing rapidly with foundation models, agentic systems, and multimodal understanding leading the way. Amazon Nova and similar models are making these capabilities accessible to developers and researchers worldwide."
    }
}

def get_demo_response(query: str, response_type: str) -> str:
    """Get a simulated Nova response for demo mode."""
    responses = DEMO_RESPONSES[response_type]
    query_lower = query.lower()
    for key in responses:
        if key in query_lower:
            return responses[key]
    return responses["default"]


# ============================================================================
# AGENT TOOLS — Actions the agent can take
# ============================================================================

class ResearchTools:
    """All the tools our agent can use. Think of tools as actions the agent can take."""

    @staticmethod
    def search_web(query: str) -> str:
        """Search the internet for information (simulated — connect to real API in production)."""
        search_results = {
            "ai trends": "Recent AI breakthroughs include: Foundation models improving reasoning, multimodal AI advancing vision-language tasks, agentic AI systems automating complex workflows.",
            "machine learning": "ML is a subset of AI focusing on algorithms that learn from data without explicit programming. Current trends include foundation models, transfer learning, and agentic systems.",
            "deep learning": "Deep learning uses neural networks with multiple layers to process complex patterns in data.",
            "default": f"Search results for: {query} — AI technologies continue advancing rapidly with foundation models leading innovation."
        }
        result = search_results.get(query.lower(), search_results["default"])
        return f"Web Search Result:\n{result}"

    @staticmethod
    def fetch_document(doc_name: str) -> str:
        """Fetch and read documents (simulated — connect to S3/storage in production)."""
        documents = {
            "research_paper": "Title: The Future of Agentic AI\nContent: Agents autonomously complete tasks...",
            "report": "Annual Tech Report 2025: Market trends show 45% growth in AI adoption...",
        }
        return documents.get(doc_name.lower(), f"Document '{doc_name}' not found.")

    @staticmethod
    def analyze_data(data_type: str) -> str:
        """Perform data analysis (simulated — connect to real data sources in production)."""
        analyses = {
            "market": "Market Analysis: The AI market is projected to reach $1.8T by 2030, with agentic AI being the fastest-growing segment.",
            "sentiment": "Sentiment Analysis: 78% positive mentions of AI in recent news, with foundation models receiving the most attention.",
        }
        return analyses.get(data_type.lower(), f"Analysis for '{data_type}' complete.")

    @staticmethod
    def analyze_pdf(pdf_path: str) -> str:
        """Read and analyze real PDF documents using PyPDF2."""
        try:
            if not os.path.exists(pdf_path):
                return f"Error: PDF file '{pdf_path}' not found."

            pdf_text = ""
            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                num_pages = len(pdf_reader.pages)

                for page_num in range(num_pages):
                    page = pdf_reader.pages[page_num]
                    pdf_text += f"\n--- PAGE {page_num + 1} ---\n"
                    page_text = page.extract_text()
                    if page_text:
                        pdf_text += page_text
                    else:
                        pdf_text += "(No extractable text on this page)"

            if not pdf_text.strip():
                return f"PDF '{pdf_path}' appears to be image-based. Text extraction returned empty."

            max_chars = 4000
            truncated = len(pdf_text) > max_chars
            display_text = pdf_text[:max_chars] if truncated else pdf_text

            summary = f"""\nPDF Analysis Summary:\n- File: {pdf_path}\n- Pages: {num_pages}\n- Content: {len(pdf_text)} characters extracted\n\nExtracted Text:\n{display_text}\n{"... [truncated]" if truncated else ""}\n\nFull text available for processing by Nova 2 Lite.\n"""
            return summary

        except Exception as e:
            return f"Error analyzing PDF: {str(e)}"


# ============================================================================
# THE AGENT — The brain that reasons, decides, and acts
# ============================================================================

class ResearchAgent:
    """
    The AGENT — it reasons about problems using Amazon Nova 2 Lite,
    decides which tools to use, executes them, and synthesizes answers.
    """

    def __init__(self):
        self.conversation_history = []
        self.tools = {
            "search_web": ResearchTools.search_web,
            "fetch_document": ResearchTools.fetch_document,
            "analyze_data": ResearchTools.analyze_data,
            "analyze_pdf": ResearchTools.analyze_pdf,
        }

    def format_tools_for_prompt(self) -> str:
        """Tell Nova what tools are available."""
        return """
Available Tools:
1. search_web(query) - Search the internet for information
2. fetch_document(doc_name) - Read a document
3. analyze_data(data_type) - Perform analysis on data
4. analyze_pdf(pdf_path) - Read and analyze PDF documents

When responding, if you need to use a tool, format it as:
ACTION: [tool_name]
INPUT: [what to search for or filename]
"""

    def call_nova_model(self, prompt: str) -> str:
        """Send a prompt to Amazon Nova 2 Lite via AWS Bedrock Converse API.
        Includes retry logic for throttling errors."""

        # If demo mode is on, skip the API call
        if USE_DEMO_MODE:
            return None  # Handled by caller

        for attempt in range(MAX_RETRIES):
            try:
                # IMPORTANT: content must be a LIST of content blocks
                message = {
                    "role": "user",
                    "content": [{"text": prompt}]
                }

                client = get_bedrock_client()
                response = client.converse(
                    modelId=MODEL_ID,
                    messages=[message],
                    inferenceConfig={
                        "maxTokens": 1024,
                        "temperature": 0.7,
                    }
                )

                nova_response = response['output']['message']['content'][0]['text']
                return nova_response

            except Exception as e:
                error_str = str(e)
                if 'ThrottlingException' in error_str or 'Too many tokens' in error_str:
                    if attempt < MAX_RETRIES - 1:
                        wait_time = RETRY_WAIT * (attempt + 1)
                        print(f"   ⏳ API throttled. Waiting {wait_time}s before retry {attempt + 2}/{MAX_RETRIES}...")
                        time.sleep(wait_time)
                    else:
                        print(f"   ⚠️ API quota exceeded after {MAX_RETRIES} retries.")
                        print(f"   💡 Tip: Set USE_DEMO_MODE = True in Step 2 to use simulated responses.")
                        return f"Error: API throttled — {error_str}"
                else:
                    return f"Error calling Nova model: {error_str}"

        return "Error: Max retries exceeded."

    def reason_and_decide(self, user_query: str) -> dict:
        """Use Nova to reason about the question and decide what tool to use."""

        if USE_DEMO_MODE:
            # Use pre-built demo response
            demo_reasoning = get_demo_response(user_query, "reasoning")
            return {
                "reasoning": demo_reasoning,
                "action": self.extract_action(demo_reasoning),
                "input": self.extract_input(demo_reasoning),
            }

        reasoning_prompt = f"""
You are a smart research assistant agent. A user asked you:
\"{user_query}\"

{self.format_tools_for_prompt()}

Your task:
1. Break down what the user is asking
2. Decide which tools you need
3. Explain your reasoning

Format your response like this:
THOUGHT: [What you're thinking]
ACTION: [Which tool to use]
INPUT: [What to search/fetch/analyze]
FINAL_ANSWER: [Your conclusion after research]
"""
        nova_response = self.call_nova_model(reasoning_prompt)
        return {
            "reasoning": nova_response,
            "action": self.extract_action(nova_response),
            "input": self.extract_input(nova_response),
        }

    def extract_action(self, response: str) -> str:
        """Extract which tool the agent wants to use (handles LLM formatting variations)."""
        if "ACTION:" in response:
            action_lines = [line for line in response.split('\n') if 'ACTION:' in line]
            if not action_lines:
                return "search_web"
            raw_action = action_lines[0].split("ACTION:")[1].strip().lower()
            raw_action = re.split(r'[\(\s]', raw_action)[0]
            for tool_name in self.tools:
                if tool_name in raw_action:
                    return tool_name
            return raw_action
        return "search_web"

    def extract_input(self, response: str) -> str:
        """Extract the input parameter for the selected tool."""
        if "INPUT:" in response:
            input_lines = [line for line in response.split('\n') if 'INPUT:' in line]
            if not input_lines:
                return ""
            raw_input = input_lines[0].split("INPUT:")[1].strip()
            raw_input = raw_input.strip('"').strip("'")
            return raw_input
        return ""

    def execute_action(self, action: str, input_data: str) -> str:
        """Execute the tool that the agent decided to use."""
        if action in self.tools:
            return self.tools[action](input_data)
        return "Tool not found."

    def run(self, user_query: str) -> str:
        """
        Main agent loop:
        1. Understand the question
        2. Reason about it using Nova
        3. Decide what tools to use
        4. Execute the tools
        5. Synthesize the final answer
        """
        print(f"\n{'='*70}")
        print(f"USER QUERY: {user_query}")
        print(f"{'='*70}\n")

        mode_label = "DEMO MODE" if USE_DEMO_MODE else "LIVE — Amazon Nova 2 Lite"
        print(f"[Mode: {mode_label}]\n")

        # STEP 1: Reason
        print("🤔 Agent is reasoning about your question...")
        decision = self.reason_and_decide(user_query)
        print(f"\nAgent's Reasoning:\n{decision['reasoning']}\n")

        # STEP 2: Decide
        action = decision['action']
        input_data = decision['input']

        if action and input_data:
            print(f"🔧 Agent decided to use: {action.upper()}")
            print(f"   With input: {input_data}\n")

            # STEP 3: Execute
            print("⚙️  Executing action...")
            tool_result = self.execute_action(action, input_data)
            print(f"Tool Result:\n{tool_result}\n")

            # STEP 4: Synthesize
            print("✨ Agent is synthesizing final answer...\n")

            if USE_DEMO_MODE:
                final_answer = get_demo_response(user_query, "synthesis")
            else:
                synthesis_prompt = f"""
Based on this research:
- Original Question: {user_query}
- Tool Used: {action}
- Information Found: {tool_result}

Please provide a comprehensive, clear answer to the original question.
Use the information found to support your answer.
"""
                final_answer = self.call_nova_model(synthesis_prompt)

            print(f"{'='*70}")
            print("FINAL ANSWER:")
            print(f"{'='*70}")
            print(final_answer)
            return final_answer
        else:
            print("⚠️  Agent could not determine what to do.")
            return "Unable to process your request."


print("✅ Agent code loaded successfully!")
print("   Tools available: search_web, fetch_document, analyze_data, analyze_pdf")
if USE_DEMO_MODE:
    print("   ⚡ Running in DEMO MODE (simulated Nova responses)")
else:
    print("   🔗 Running in LIVE MODE (real Amazon Nova API calls)")

## Step 4: Create the Agent

In [ ]:
# Create the agent
agent = ResearchAgent()

print("🚀 SMART RESEARCH ASSISTANT AGENT  #AmazonNova")
print("   Powered by Amazon Nova 2 Lite on AWS Bedrock")
print()
print("   Available tools:")
for name in agent.tools:
    print(f"     • {name}")
print()
print("✅ Agent is ready!")

---

## 🎬 DEMO 1: Web Research

The agent reasons about a research question, decides to use web search, and synthesizes an answer.

In [ ]:
# DEMO 1: Ask the agent a research question
agent.run("What is the current state of agentic AI systems?")

## 🎬 DEMO 2: PDF Document Analysis

First, let's create a sample PDF. Then the agent reads it with PyPDF2 and analyzes the content.

In [ ]:
# Create a sample research paper PDF for testing
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

c = canvas.Canvas('sample.pdf', pagesize=letter)
c.setFont('Helvetica-Bold', 16)
c.drawString(72, 720, 'The Future of Agentic AI Systems')
c.setFont('Helvetica', 12)
c.drawString(72, 690, 'A Research Overview')
c.drawString(72, 660, '')
c.drawString(72, 640, 'Abstract: This paper explores the emerging field of agentic AI,')
c.drawString(72, 620, 'where autonomous systems can plan, reason, and take actions')
c.drawString(72, 600, 'to accomplish complex tasks without human intervention.')
c.drawString(72, 570, 'Introduction')
c.drawString(72, 550, 'Agentic AI represents a paradigm shift in artificial intelligence.')
c.drawString(72, 530, 'Unlike traditional AI that responds to single prompts, agents can')
c.drawString(72, 510, 'break down problems, use tools, and iteratively work toward goals.')
c.drawString(72, 480, 'Key Findings')
c.drawString(72, 460, '1. Foundation models like Amazon Nova enable sophisticated reasoning.')
c.drawString(72, 440, '2. Tool use is critical for agents to interact with the real world.')
c.drawString(72, 420, '3. Multi-step planning improves task completion by 45 percent.')
c.drawString(72, 390, 'Conclusion')
c.drawString(72, 370, 'Agentic AI powered by foundation models will transform workflows')
c.drawString(72, 350, 'across industries, from research to software development.')
c.save()

print("✅ sample.pdf created successfully!")
print("   The agent will now read this real PDF file using PyPDF2.")

In [ ]:
# DEMO 2: Agent reads and analyzes the real PDF
agent.run("Analyze the key findings in sample.pdf")

## 🎬 DEMO 3: Data Analysis

The agent recognizes this is a data question and uses the analyze_data tool.

In [ ]:
# DEMO 3: Market analysis question
agent.run("What does market analysis say about AI growth?")

## 🎬 DEMO 4: Try Your Own Question!

In [ ]:
# Try your own question here!
agent.run("Tell me about machine learning trends")

---

## 📋 How This Agent Uses Amazon Nova

| Step | What Happens | Amazon Nova's Role |
|------|-------------|-------------------|
| 1. Reasoning | Agent analyzes user question | Nova 2 Lite decides the best approach |
| 2. Tool Selection | Agent picks the right tool | Nova outputs THOUGHT → ACTION → INPUT |
| 3. Execution | Tool runs and returns data | (Tools handle this step) |
| 4. Synthesis | Raw data → clear answer | Nova produces comprehensive response |

**Model:** `amazon.nova-lite-v1:0` via AWS Bedrock Converse API

---

**#AmazonNova** | Amazon Nova AI Hackathon 2026 | Agentic AI Category | Built by Devika